In [2]:
%load_ext autoreload
%autoreload 2


import sys
import os

# Get the absolute path to the project root directory (the parent of 'notebooks')
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to sys.path so Python can find the 'src' package
if project_root not in sys.path:
    sys.path.append(project_root)

# Now the import should work
from src.data_loader import EndoDataLoader

# Proceed with your code
loader = EndoDataLoader(data_path='../data/')
adata = loader.load_dataset('GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5')
print(adata)

Successfully loaded: GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5 | Cells: 4133 | Genes: 33538 | Label: 0
AnnData object with n_obs × n_vars = 4133 × 33538
    obs: 'label'
    var: 'gene_ids', 'feature_types', 'genome'


/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [3]:
# Fix duplicate gene names
adata.var_names_make_unique()
print("Duplicate var names resolved.")

Duplicate var names resolved.


In [4]:
adata = loader.load_dataset('GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5')
adata.var_names_make_unique() # הפתרון לאזהרה שראינו
adata = loader.preprocess(adata)
print(adata)

/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Successfully loaded: GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5 | Cells: 4133 | Genes: 33538 | Label: 0
Preprocessing complete. Remaining genes: 2000
AnnData object with n_obs × n_vars = 3870 × 2000
    obs: 'label', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'


In [5]:
from src.data_loader import adata_to_tensor
# נניח ש-adata הוא הדאטה המעובד שלנו
X_tensor = adata_to_tensor(adata)
print(X_tensor.shape) # צריכה לראות [3870, 2000]

torch.Size([3870, 2000])


In [6]:
import torch

# Phase B: Initialize the model
from src.models import AttentionClassifier

# We define the model structure
model = AttentionClassifier(input_dim=2000, hidden_dim=128, num_classes=2)
print("Model initialized successfully!")

# Phase C: Dry Run (Forward pass)
# We pass the tensor through the model to ensure dimensions match
# We use torch.no_grad() because we aren't training yet
with torch.no_grad():
    test_output, test_weights = model(X_tensor)

print(f"Output shape: {test_output.shape}") # Should be [3870, 2]
print(f"Attention weights shape: {test_weights.shape}") # Should be [3870, 1]

Model initialized successfully!
Output shape: torch.Size([3870, 2])
Attention weights shape: torch.Size([3870, 1])


In [7]:
# Print the available clinical metadata
print(adata.obs.head())
# Look for a column like 'condition', 'diagnosis', 'disease_state', or 'sample_type'
print(adata.obs.columns)

                    label  n_genes_by_counts  log1p_n_genes_by_counts  \
AAACCCAGTGTTGAGG-1      0                352                 5.866468   
AAACCCATCGCTCTAC-1      0               5737                 8.654866   
AAACGAAAGCCTTCTC-1      0               1024                 6.932448   
AAACGAACAGCACACC-1      0               4757                 8.467583   
AAACGAACATGGGTTT-1      0                213                 5.365976   

                    total_counts  log1p_total_counts  \
AAACCCAGTGTTGAGG-1         551.0            6.313548   
AAACCCATCGCTCTAC-1       31445.0           10.356027   
AAACGAAAGCCTTCTC-1        1613.0            7.386471   
AAACGAACAGCACACC-1       16510.0            9.711782   
AAACGAACATGGGTTT-1        2064.0            7.632885   

                    pct_counts_in_top_50_genes  pct_counts_in_top_100_genes  \
AAACCCAGTGTTGAGG-1                   45.190563                    54.264973   
AAACCCATCGCTCTAC-1                   30.027031                    

In [8]:
# Convert the 'label' column from AnnData to a PyTorch Tensor
# We use dtype=torch.long because CrossEntropyLoss expects integer labels
y_tensor = torch.tensor(adata.obs['label'].values, dtype=torch.long)

print(f"Labels shape: {y_tensor.shape}")
print(f"Labels preview: {y_tensor[:10]}") # לראות את התגיות של 10 התאים הראשונים

Labels shape: torch.Size([3870])
Labels preview: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [10]:
import scanpy as sc

# 1. טעינת קובץ ביקורת (Control - תווית 0)
adata_ctrl = loader.load_dataset('GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5')
adata_ctrl.var_names_make_unique()
adata_ctrl = loader.preprocess(adata_ctrl)

# 2. טעינת קובץ מחלה (Endometriosis - תווית 1)
adata_endo = loader.load_dataset('GSM6102537_E01_EuE_filtered_feature_bc_matrix.h5')
adata_endo.var_names_make_unique()
adata_endo = loader.preprocess(adata_endo)

# 3. חיבור שני האטלסים יחד (Concatenation)
adata = sc.concat([adata_ctrl, adata_endo], axis=0)
# Fix duplicate observation (cell) names after concatenation
adata.obs_names_make_unique()
# בדיקה שיש לנו גם 0 וגם 1
print("\n--- Label Distribution ---")
print(adata.obs['label'].value_counts())

/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Successfully loaded: GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5 | Cells: 4133 | Genes: 33538 | Label: 0
Preprocessing complete. Remaining genes: 2000


/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Successfully loaded: GSM6102537_E01_EuE_filtered_feature_bc_matrix.h5 | Cells: 7665 | Genes: 33538 | Label: 1
Preprocessing complete. Remaining genes: 2000

--- Label Distribution ---
label
1    7662
0    3870
Name: count, dtype: int64


/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [13]:
import scanpy as sc
from src.data_loader import EndoDataLoader, adata_to_tensor
from src.models import AttentionClassifier
import torch
import torch.optim as optim
import torch.nn as nn

# 1. Initialize loader
loader = EndoDataLoader(data_path='../data/')

# 2. Load raw datasets separately
adata_ctrl = loader.load_dataset('GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5')
adata_ctrl.var_names_make_unique()

adata_endo = loader.load_dataset('GSM6102537_E01_EuE_filtered_feature_bc_matrix.h5')
adata_endo.var_names_make_unique()

# 3. Concatenate RAW data using 'inner' to keep common genes
adata = sc.concat([adata_ctrl, adata_endo], axis=0, join='inner')
adata.obs_names_make_unique()

# 4. Preprocess the combined dataset together (ensures exactly 2000 HVGs)
adata = loader.preprocess(adata)

# 5. Convert to Tensors
X_tensor = adata_to_tensor(adata)
y_tensor = torch.tensor(adata.obs['label'].values, dtype=torch.long)

print(f"Fixed X_tensor shape: {X_tensor.shape}") # Should be [11532, 2000]
print(f"Fixed y_tensor shape: {y_tensor.shape}")

# 6. Re-initialize model to match the exact input dimension
input_dim = X_tensor.shape[1]
model = AttentionClassifier(input_dim=input_dim, hidden_dim=128, num_classes=2)

# 7. Training Setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 8. Training Loop
num_epochs = 15
print("\nStarting training with correctly aligned dimensions...")

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    # Forward pass
    outputs, weights = model(X_tensor)

    # Calculate loss
    loss = criterion(outputs, y_tensor)

    # Backward pass & optimization
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 3 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

print("\nTraining finished successfully!")

/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Successfully loaded: GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5 | Cells: 4133 | Genes: 33538 | Label: 0


/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/dani/PycharmProjects/EndoSignature-Net./.venv/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Successfully loaded: GSM6102537_E01_EuE_filtered_feature_bc_matrix.h5 | Cells: 7665 | Genes: 33538 | Label: 1
Preprocessing complete. Remaining genes: 2000
Fixed X_tensor shape: torch.Size([11532, 2000])
Fixed y_tensor shape: torch.Size([11532])

Starting training with correctly aligned dimensions...
Epoch [3/15], Loss: 0.4470
Epoch [6/15], Loss: 0.2962
Epoch [9/15], Loss: 0.2314
Epoch [12/15], Loss: 0.2044
Epoch [15/15], Loss: 0.1801

Training finished successfully!


In [15]:
import pandas as pd
import torch

# 1. Extract the weights of the first Linear layer connecting the 2000 genes
# Shape: [hidden_dim, input_dim] -> [128, 2000]
first_layer_weights = model.feature_extractor[0].weight.abs()

# 2. Average the weights across the hidden dimensions to get a global score per gene
# Shape: [2000]
gene_importance = first_layer_weights.mean(dim=0).detach().cpu().numpy()

# 3. Map to the 2,000 highly variable gene names
gene_names = adata.var_names

# 4. Create a DataFrame to view genes alongside their importance score
gene_importance_df = pd.DataFrame({
    'Gene': gene_names,
    'Importance_Score': gene_importance
})

# 5. Sort genes by highest importance score
gene_importance_df = gene_importance_df.sort_values(by='Importance_Score', ascending=False)

# 6. Display the top 15 marker genes discovered by the model
print("Top 15 Biomarker Genes Discovered by the Model:")
print(gene_importance_df.head(15))

Top 15 Biomarker Genes Discovered by the Model:
            Gene  Importance_Score
1056     SCGB1D2          0.016562
1751      BPIFB1          0.016266
1731  AC124254.1          0.016102
139       FCGR3B          0.016039
1725    SERPINB4          0.015873
1738  AL049647.1          0.015735
1178     CYP26A1          0.015619
977        PTGS1          0.015517
1551       CMTM2          0.015479
360        FAM3D          0.015427
1031  AC111188.1          0.015396
1856      NCCRP1          0.015384
498          FGB          0.015345
1107      TRIM29          0.015307
1083       MMP10          0.015296
